# Deliverable 2

In [ ]:
# Imports

import re
import json
import pandas as pd
from pathlib import Path
from charset_normalizer import from_path
import numpy as np

#### Task 1

In [6]:
test_path = Path("data/2026/student_union/POS_01/register_sales.log")

def file_reader(file_path: Path) -> tuple[str, str]:
    """
    Reads a file checking its encoding.

    args:
        file_path: The path for the file to be checked
    
    returns:
        text: the encoded text
        encoding: the encoding of the given text

    """
    raw_bytes = Path(file_path).read_bytes()

    # Checking for utf-8 encoding with BOM
    if raw_bytes[:3] == b"\xef\xbb\xbf":
        encoding = "utf-8-sig"

    # Otherwise try to guess correct encoding
    else:
        best_match = from_path(file_path).best()
        if best_match is None:
            raise ValueError(f"Could not detect encoding of {file_path}")
        encoding = best_match.encoding

    decoded_text = raw_bytes.decode(encoding)
    return decoded_text, encoding

text, encoding = file_reader(test_path)
#print(f"Encoding: {encoding}\n{text}")

FileNotFoundError: [Errno 2] No such file or directory: 'data\\2026\\student_union\\POS_01\\register_sales.log'

For de log_parser() I've used lecture 6 for regex syntax

In [ ]:
def log_parser(decoded_text: str) -> list[dict][str, str | float]:
    """
    Uses regular expression to capture 6 key fields from valid lines in a log file and
    ignores noise lines.

    Key fields:
        timestamp
        register_id
        item
        price
        currency
        status
    
    args:
        decoded_text: a string of from decoded file.

    returns: 
        list: List of dictonaries with valid transaction record
    """

    verbose_pattern = re.compile(
        r"""                                   
        \[(?P<timestamp> \d{4}-\d{2}-\d{2}\ \d{2}:\d{2}:\d{2})\]    # datetime YYYY-MM-DD HH:MM:SS
        \s+                                                         # whitespace
        \[(?P<register_id> POS_\d{2})\]                             # POS_xx
        \s+                                                         # whitespace
        item=(?P<item>\S+)                                          # item
        \s+                                                         # whitespace
        price=(?P<price>\S+)                                        # price
        \s+                                                         # whitespace
        (?P<currency>[A-Za-z]{3})                                   # currency
        \s+                                                         # whitespace
        \( (?P<status>[\w-]+) \)                                    # status
        $                                   
    """,
        re.VERBOSE,
    )    

    valid_records = []
    for line in decoded_text.splitlines():
        m = verbose_pattern.match(line)

        # check for non-valid lines
        if not m:
            continue

        valid_records.append(
            {
            "timestamp": m.group("timestamp"),
            "register_id": m.group("register_id"),
            "item": m.group("item"),
            "price": float(m.group("price")), # converts str to float here
            "currency": m.group("currency"),
            "status": m.group("status"),
            }
        )

    return valid_records

log_parser(text)

[{'timestamp': '2026-10-15 07:02:16',
  'register_id': 'POS_01',
  'item': 'Croissant',
  'price': 4.0,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:04:24',
  'register_id': 'POS_01',
  'item': 'Muffin',
  'price': 3.38,
  'currency': 'EUR',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:04:35',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.94,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:04',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.96,
  'currency': 'USD',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:05:14',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.79,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:06:43',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.2,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:09:14',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 4.23,
  

#### Task 2

In [ ]:
# test path
test_path = ("data/2026/library_cafe")

def recursive_dir_search(input_path: Path | str) -> list[Path]:
    """
    Recursively search through a directory hierarchy finding .log files.

    arg:
        input_path: This can be a path object or a string for the location for where
        to start the search.

    return:
        path_list: A list with the path to the found .log files.
    """
    inp_pat = Path(input_path)

    path_list = []
    for i in inp_pat.iterdir():
        if i.is_dir():
            path_list.extend(recursive_dir_search(i))
        elif i.is_file() and i.suffix == ".log":
            path_list.append(i)

    return path_list

log_files = recursive_dir_search(test_path)
print(log_files)

[WindowsPath('data/2026/library_cafe/POS_01/register_sales.log'), WindowsPath('data/2026/library_cafe/POS_02/register_sales.log'), WindowsPath('data/2026/library_cafe/POS_03/register_sales.log')]


In [ ]:
test_path_json = ("data/store_config.json")

def traverse_nested(data: dict | list) -> set[str]:

    """
    Goes through a nested file of dicts and lists,
    recursivly checking for strings where "active": true.

    arg:
        data: parsed json data

    return:
        active_id: a set of register id strings where "active": true

    """
    active_id = set()

    if isinstance(data, dict):
        if data.get("active") is True and isinstance(data.get("id"), str):
            active_id.add(data["id"])

        for value in data.values():
            active_id.update(traverse_nested(value))

    elif isinstance(data, list):
        for item in data:
            active_id.update(traverse_nested(item))

    return active_id

with open(test_path_json) as f:
    config = json.load(f)

traverse_nested(config)

{'POS_01', 'POS_02', 'POS_03'}

#### Task 3

In [ ]:
def remove_price_outliers(prices: list[float] | np.ndarray) -> np.ndarray:
    """




    """

#### Task 4

#### Task 5